# From a Simulated Pattern to a Measured One: Background, Instrument, and Rietveld Refinement

Tutorial 11 simulated a powder pattern. It computed where the peaks are, how tall they are, and
which $(hkl)$ each belongs to. That is a complete answer to *"what would this phase diffract?"*
and no answer at all to the four questions a diffractometer actually raises:

1. **What is the background?** A raw scan is Bragg intensity plus air scatter, fluorescence,
   incoherent scattering, the holder and the detector. Every integrated intensity depends on
   which is which.
2. **How much of the peak width is the instrument?** A crystallite size read off an
   uncorrected width is not an underestimate of the truth. It is a measurement of the
   diffractometer.
3. **Does the model actually account for the data?** Not "do the peak positions look about
   right" but a number, computed on every point of the profile, that can be compared against
   the counting noise.
4. **What does the pattern say the cell, the width, and the texture actually are?**

This tutorial answers all four, in that order, because each one depends on the previous. It uses
a *synthetic* measurement throughout — built from a known cell, a known zero-point error, a known
peak width and a known texture, then given Poisson counting noise. That is the point: with ground
truth in hand, every step can be checked rather than admired. A tutorial on refinement that only
shows a nice-looking fit teaches the one habit that makes refinement dangerous.

**What you will be able to do afterwards.** Estimate a background from a raw scan and know which
estimator to reach for; calibrate an instrumental resolution function from a standard and
deconvolve it; separate crystallite size from microstrain across several reflections; run a
whole-profile refinement and read its agreement indices honestly, including the cases where a
good $R_{wp}$ means nothing.

**What this tutorial does not do.** It does not determine a structure. PyTex refines the profile,
the cell dilation, the zero point, the texture strength and the background against a *known*
structural model; atomic coordinates, occupancies and anisotropic displacement parameters are
deliberately outside its scope, for reasons taken up in the last section.

## 0. Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import warnings

warnings.filterwarnings("ignore", message="Issues encountered while parsing CIF")
warnings.filterwarnings("ignore", message="No _symmetry_equiv_pos_as_xyz")

from pytex import (
    FrameDomain,
    InstrumentBroadening,
    MeasuredPowderPattern,
    RadiationSpec,
    ReferenceFrame,
    calibrate_instrument_broadening,
    compare_powder_patterns,
    deconvolve_instrument_width,
    estimate_background,
    generate_xrd_pattern,
    get_phase_fixture,
    refine_rietveld,
    scherrer_size_nm,
    williamson_hall,
)
from pytex.core.miller import MillerPlane
from pytex.diffraction.preferred_orientation import MarchDollaseModel
from pytex.diffraction.rietveld import _scaled_phase

plt.rcParams["figure.figsize"] = (9.0, 4.0)
plt.rcParams["figure.dpi"] = 110

CRYSTAL = ReferenceFrame("crystal", FrameDomain.CRYSTAL, ("a", "b", "c"))
RADIATION = RadiationSpec.cu_ka()

# The pinned fixture corpus, not a hand-typed cell: the CIF is hash-checked, so
# every number below is reproducible from a file in the repository.
FIXTURE = get_phase_fixture("ni_fcc")
NICKEL = FIXTURE.load_phase(crystal_frame=CRYSTAL)
print(f"{NICKEL.name}: a = {NICKEL.lattice.a:.5f} A, {NICKEL.space_group_symbol}")
print(f"fixture {FIXTURE.fixture_id}, CIF sha256 {FIXTURE.artifact_sha256[:16]}...")
print(f"radiation: {RADIATION.name}, lambda = {RADIATION.wavelength_angstrom:.6f} A")

## 1. A measurement with ground truth

Everything below is checked against the numbers this cell chooses. The pattern is built from
nickel with four deliberate departures from the ideal:

| Departure | Value | Why it belongs here |
| --- | --- | --- |
| Cell dilation | $\times 1.003$ | A real specimen is never at the tabulated cell — thermal expansion, solid solution, or the wrong reference |
| Detector zero error | $+0.05^\circ$ | The classic instrument misalignment, and the classic thing to mistake for a cell change |
| Peak width | $0.14^\circ$ FWHM | Wider than the ideal instrument the refinement will start from |
| Background | curved, ~150 counts | Not flat, because a flat background is the one case where the choice of estimator does not matter |

Then Poisson noise, because the weights a refinement uses are counting weights and a noise-free
pattern would make the goodness of fit meaningless.

In [ ]:
TRUE_LATTICE_SCALE = 1.003
TRUE_ZERO_SHIFT_DEG = 0.05
TRUE_FWHM_DEG = 0.14

truth_phase = _scaled_phase(NICKEL, TRUE_LATTICE_SCALE)
ideal = generate_xrd_pattern(
    truth_phase,
    radiation=RADIATION,
    two_theta_range_deg=(30.0, 130.0),
    resolution_deg=0.02,
    broadening_fwhm_deg=TRUE_FWHM_DEG,
    intensity_model="xray_tabulated",
)

angles = ideal.two_theta_grid_deg + TRUE_ZERO_SHIFT_DEG
noiseless = (
    20000.0 * ideal.intensity_grid
    + 150.0
    + 400.0 * np.exp(-0.5 * ((angles - 34.0) / 6.0) ** 2)
)
counts = np.random.default_rng(20260905).poisson(noiseless).astype(float)

measured = MeasuredPowderPattern(
    name="nickel, synthetic laboratory scan",
    two_theta_deg=angles,
    intensity=counts,
    radiation=RADIATION,
    synthetic=True,
)
print(f"true cell:  a = {truth_phase.lattice.a:.6f} A")
print(f"tabulated:  a = {NICKEL.lattice.a:.6f} A")
print(f"{measured.two_theta_deg.size} points, "
      f"{measured.two_theta_deg[0]:.2f} to {measured.two_theta_deg[-1]:.2f} deg 2theta")
print(f"peak counts {counts.max():.0f}, background about {np.median(counts):.0f}")

In [ ]:
fig, ax = plt.subplots()
ax.plot(measured.two_theta_deg, measured.intensity, lw=0.8, color="#1f4e79")
ax.set_xlabel(r"$2\theta$ (deg)")
ax.set_ylabel("counts")
ax.set_title("The measurement everything below works from")
ax.set_xlim(30.0, 130.0)
fig.tight_layout()

## 2. The background

Two estimators, because they answer different questions.

**SNIP** — Statistics-sensitive Non-linear Iterative Peak-clipping (Ryan *et al.* 1988) — clips
each point against the mean of its neighbours at a growing separation, in a
$\log\log\sqrt{\;}$ domain that makes the clip insensitive to count level. It assumes *only* that
the background varies more slowly with angle than the peaks do, so it follows curved and
structured backgrounds that no low-order polynomial can. Reach for it on unfamiliar data, and to
*see* what the background is doing.

**Chebyshev** — a polynomial fitted by iteratively reweighted least squares, discarding points
that sit more than `clip_sigma` residual standard deviations *above* the current curve. The
asymmetry is the whole design: peaks are one-sided excursions, so a symmetric rejection would
drag the curve up into the peak feet. It gives a handful of coefficients a refinement can carry,
which is why Rietveld programs use this family.

Neither estimator is told where the peaks are. That is deliberate — a background that was told
where the peaks are cannot be used to *find* them, and the failure would be silent.

The one parameter that matters for SNIP is the clipping window. It must be comfortably wider than
the broadest peak and comfortably narrower than the curvature of the background.

In [ ]:
snip = estimate_background(measured, method="snip", half_window_deg=2.0)
chebyshev = estimate_background(measured, method="chebyshev", degree=6)

print(snip.describe())
print()
print(chebyshev.describe())

In [ ]:
truth_background = 150.0 + 400.0 * np.exp(-0.5 * ((angles - 34.0) / 6.0) ** 2)

fig, (top, bottom) = plt.subplots(2, 1, figsize=(9.0, 6.0), sharex=True,
                                  gridspec_kw={"height_ratios": [2, 1]})
top.plot(measured.two_theta_deg, measured.intensity, lw=0.6, color="#b0b0b0", label="measured")
top.plot(angles, truth_background, lw=2.0, color="#111111", label="true background")
top.plot(snip.two_theta_deg, snip.background, lw=1.4, color="#c0392b", label="SNIP")
top.plot(chebyshev.two_theta_deg, chebyshev.background, lw=1.4, color="#1f4e79",
         label="Chebyshev, degree 6")
top.set_ylabel("counts")
top.set_ylim(0.0, 1200.0)
top.legend(loc="upper right", fontsize=8)
top.set_title("Two estimators against the background they are trying to find")

bottom.axhline(0.0, color="#111111", lw=1.0)
bottom.plot(angles, snip.background - truth_background, lw=1.0, color="#c0392b", label="SNIP")
bottom.plot(angles, chebyshev.background - truth_background, lw=1.0, color="#1f4e79",
            label="Chebyshev")
bottom.set_xlabel(r"$2\theta$ (deg)")
bottom.set_ylabel("error (counts)")
bottom.legend(loc="upper right", fontsize=8)
fig.tight_layout()

for name, estimate in (("SNIP", snip), ("Chebyshev", chebyshev)):
    error = estimate.background - truth_background
    print(f"{name:10s} mean |error| {np.abs(error).mean():7.2f} counts, "
          f"worst {np.abs(error).max():7.2f}, "
          f"background fraction {estimate.background_fraction:.3f}")

Both follow the hump. SNIP is the more faithful where the background curves, and both are wrong
by a few counts at the peak feet — which is exactly where a background estimate is least
constrained, because there is no measurement of the background there at all, only an
extrapolation between the gaps.

Subtracting is a modelling act, and the result records it, so a subtracted pattern can never be
mistaken for a raw one.

In [ ]:
subtracted = snip.subtract(measured)
print(subtracted.name)
print(dict(subtracted.metadata))
print(f"minimum intensity after subtraction: {subtracted.intensity.min():.1f} (clipped at zero)")

**But do not do this before a refinement.** Section 6 refines the background *jointly* with
everything else. Subtracting first throws away the correlation between background and scale, and
the standard uncertainties a refinement reports depend on exactly that correlation. Estimate the
background to look at it; refine it to use it.

## 3. The instrument

A peak is wider than the sample makes it. `InstrumentBroadening` states how much wider, as a
function of angle: a Caglioti Gaussian width

$$\mathrm{FWHM}_G^2 = U\tan^2\theta + V\tan\theta + W$$

plus a Lorentzian term $\mathrm{FWHM}_L = X\tan\theta + Y/\cos\theta$, combined into one
pseudo-Voigt by the Thompson–Cox–Hastings construction. The two are *combined*, not added,
because the convolution of a Gaussian with a Lorentzian is neither.

The honest way to get $U$, $V$, $W$ is to measure a standard — NIST SRM 660 LaB$_6$, SRM 640
silicon, anything whose own broadening is negligible against the instrument's. Because the
Caglioti relation is *linear* in $U$, $V$ and $W$ once you work in squared widths, that fit is an
exact linear least squares: no starting values, no convergence question.

In [ ]:
standard_angles = np.array([21.4, 30.4, 37.5, 43.6, 53.9, 67.6, 79.3, 95.2, 115.4])
reference = InstrumentBroadening(caglioti_u=0.0035, caglioti_v=-0.0021, caglioti_w=0.0060)
standard_widths = reference.gaussian_fwhm_deg(standard_angles)

instrument = calibrate_instrument_broadening(
    standard_angles, standard_widths, name="SRM 660c LaB6 calibration"
)
print(instrument.describe())
print()
print("recovered exactly from the widths it was built from:")
for label, fitted, expected in (
    ("U", instrument.caglioti_u, 0.0035),
    ("V", instrument.caglioti_v, -0.0021),
    ("W", instrument.caglioti_w, 0.0060),
):
    print(f"  {label} = {fitted:+.8f}  (expected {expected:+.8f})")

A caution the API enforces rather than documents. A parabola through a handful of widths can dip
below zero *between* them, and the failure is silent: the object constructs, and only a width
evaluation months later raises. `calibrate_instrument_broadening` therefore evaluates its own fit
across the calibrated range before returning, so a badly conditioned calibration fails at the
moment when the fix — measure more standard peaks — is still the obvious one.

## 4. Size and strain

With the instrument known, a measured width can be reduced to the sample's own. What is left
contains two things that no single peak can separate, because both widen the peak. Their
*angular dependences* differ, and that is the whole of the Williamson–Hall method:

$$\beta\cos\theta = \frac{K\lambda}{D} + 4\varepsilon\sin\theta$$

Size sits in the intercept ($1/\cos\theta$ behaviour), strain in the slope ($\tan\theta$
behaviour). Below, a specimen with a known 25 nm crystallite size and a known 0.2% microstrain is
measured on the calibrated instrument, and both are recovered.

In [ ]:
TRUE_SIZE_NM = 25.0
TRUE_MICROSTRAIN = 0.002

theta = np.deg2rad(0.5 * standard_angles)
sample_width = np.rad2deg(
    0.9 * RADIATION.wavelength_angstrom / (10.0 * TRUE_SIZE_NM) / np.cos(theta)
    + 4.0 * TRUE_MICROSTRAIN * np.tan(theta)
)
observed_width = np.sqrt(sample_width**2 + instrument.fwhm_deg(standard_angles) ** 2)

recovered = deconvolve_instrument_width(
    observed_width, instrument, standard_angles, mode="gaussian"
)
analysis = williamson_hall(
    standard_angles, recovered, wavelength_angstrom=RADIATION.wavelength_angstrom
)
print(analysis.describe())

In [ ]:
scherrer = scherrer_size_nm(
    recovered, standard_angles, wavelength_angstrom=RADIATION.wavelength_angstrom
)
print(f"Williamson-Hall size : {analysis.crystallite_size_nm:8.3f} nm  (true {TRUE_SIZE_NM})")
print(f"Williamson-Hall strain: {analysis.microstrain:8.6f}     (true {TRUE_MICROSTRAIN})")
print()
print("Scherrer, peak by peak -- every one an underestimate, and they disagree with")
print("each other, which is the signature of strain the Scherrer relation cannot express:")
for angle, size in zip(standard_angles, scherrer, strict=True):
    print(f"  {angle:6.1f} deg -> {size:6.2f} nm")

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 4.0))
ax.scatter(analysis.abscissa, analysis.ordinate, s=36, color="#1f4e79", zorder=3, label="reflections")
line = np.linspace(0.0, float(analysis.abscissa.max()) * 1.05, 50)
ax.plot(line, analysis.slope * line + analysis.intercept, color="#c0392b", lw=1.6,
        label=f"fit, $R^2$ = {analysis.r_squared:.5f}")
ax.scatter([0.0], [analysis.intercept], marker="s", s=40, color="#c0392b", zorder=4)
ax.annotate(
    f"intercept = $K\\lambda/D$\n$D$ = {analysis.crystallite_size_nm:.2f} nm",
    xy=(0.0, analysis.intercept), xytext=(0.35, analysis.intercept * 1.55),
    arrowprops={"arrowstyle": "->", "color": "#c0392b"}, fontsize=9, color="#c0392b",
)
ax.set_xlabel(r"$4\sin\theta$")
ax.set_ylabel(r"$\beta\cos\theta$ (rad)")
ax.set_xlim(left=0.0)
ax.set_title("Williamson-Hall: size in the intercept, strain in the slope")
ax.legend(loc="lower right", fontsize=8)
fig.tight_layout()

The deconvolution refuses to do something impossible. A measured width narrower than the
instrument's own at that angle is not an extremely large crystallite — it is a measurement that
cannot have come from this instrument, so the calibration does not apply to this data.

In [ ]:
try:
    deconvolve_instrument_width(np.array([0.02]), instrument, np.array([40.0]))
except ValueError as error:
    print(f"refused: {error}")

## 5. Comparison is not refinement

Before refining, it is worth seeing what a *comparison* gives, because the two are routinely
confused and only one of them adjusts the model.

`compare_powder_patterns` interpolates a simulated profile onto the measured angles and fits one
scale and one constant background. Nothing moves. It is the right tool for phase identification —
*is this the phase?* — and the wrong tool for anything quantitative, because a peak in the wrong
place stays in the wrong place.

In [ ]:
tabulated_simulation = generate_xrd_pattern(
    NICKEL,
    radiation=RADIATION,
    two_theta_range_deg=(30.0, 130.0),
    resolution_deg=0.02,
    broadening_fwhm_deg=0.10,
    intensity_model="xray_tabulated",
)
comparison = compare_powder_patterns(measured, tabulated_simulation)
print(comparison.describe())

$R_{wp}$ around 75% on data that *is* nickel, and a Pearson correlation of only 0.6. The phase is
right; the model is wrong in three specific ways — the cell, the zero point, and the width — and
every peak is therefore in the wrong place by enough to miss its measured partner almost entirely.
A comparison has no way to say any of that. It reports a poor fit and stops, which is the honest
thing for it to do and exactly why the next section exists.

Note the trap this sets. A poor comparison does **not** mean the phase is absent — here it is the
only phase present. Reading a scale-only comparison as phase identification, without asking
whether the peaks could be brought into coincidence by a plausible cell and zero point, is one of
the reliable ways to reject a correct answer.

## 6. Refinement

`refine_rietveld` fits the whole profile. Every parameter it varies is a statement about a
distinct piece of physics, so the output says which part of the model is being adjusted:

| Parameter | Physics |
| --- | --- |
| `scale` | Overall intensity scale — always refined |
| `zero_shift_deg` | Detector zero error: an *instrument* misalignment |
| `lattice_scale` | Isotropic cell dilation, symmetry-preserving in every crystal system |
| `caglioti_u/v/w`, `lorentzian_y` | Peak width against angle |
| `b_iso_overall` | One isotropic displacement parameter shared by all sites |
| `march_coefficient` | March–Dollase texture strength about a stated axis |
| `background_0..N` | Chebyshev background, refined *jointly* |

Note what refining `zero_shift_deg` buys. A zero-point error and a cell dilation both move every
peak, so if the zero point is held at nought the cell will silently absorb it and return a
confidently wrong lattice parameter with a small uncertainty attached. Refining both is what makes
the cell mean something.

The background degree below is 10 rather than the default 4, and the reason is worth stating
because it is how the choice is actually made. This scan carries a broad hump at low angle; a
degree-4 Chebyshev is too stiff to follow it, and the misfit that leaves behind is not visible in
$R_{wp}$ (10.6% against 6.2%, which one might shrug at) but is unmistakable in the Durbin-Watson
statistic, which falls to 0.63. Section 7 takes up why that is the number to watch.

In [ ]:
result = refine_rietveld(
    measured,
    NICKEL,
    radiation=RADIATION,
    instrument=InstrumentBroadening.ideal(0.10),
    refine=("scale", "zero_shift_deg", "lattice_scale", "caglioti_w"),
    background_degree=10,
    intensity_model="xray_tabulated",
)
print(result.describe())

In [ ]:
print(f"{'parameter':<18}{'refined value':>20}{'started at':>14}{'moved by':>14}")
print("-" * 66)
for parameter in result.parameters:
    mark = "" if parameter.refined else "  (fixed)"
    print(f"{parameter.name:<18}{parameter.format():>20}"
          f"{parameter.initial_value:>14.5g}{parameter.shift:>14.5g}{mark}")

In [ ]:
recovered_scale = result.parameter("lattice_scale").value
recovered_zero = result.parameter("zero_shift_deg").value
recovered_w = result.parameter("caglioti_w").value

print(f"{'quantity':<22}{'refined':>14}{'true':>14}{'error':>14}")
print("-" * 64)
print(f"{'cell dilation':<22}{recovered_scale:>14.6f}{TRUE_LATTICE_SCALE:>14.6f}"
      f"{recovered_scale - TRUE_LATTICE_SCALE:>14.2e}")
print(f"{'zero shift (deg)':<22}{recovered_zero:>14.6f}{TRUE_ZERO_SHIFT_DEG:>14.6f}"
      f"{recovered_zero - TRUE_ZERO_SHIFT_DEG:>14.2e}")
print(f"{'W (deg^2)':<22}{recovered_w:>14.6f}{TRUE_FWHM_DEG**2:>14.6f}"
      f"{recovered_w - TRUE_FWHM_DEG**2:>14.2e}")
print(f"{'a (angstrom)':<22}{result.phase.lattice.a:>14.6f}{truth_phase.lattice.a:>14.6f}"
      f"{result.phase.lattice.a - truth_phase.lattice.a:>14.2e}")

assert abs(recovered_scale - TRUE_LATTICE_SCALE) < 5e-5
assert abs(recovered_zero - TRUE_ZERO_SHIFT_DEG) < 5e-3
assert abs(result.phase.lattice.a - truth_phase.lattice.a) < 5e-4
print("\nevery quantity recovered inside tolerance")

The lattice parameter comes back within a few parts in $10^5$ of the cell the pattern was built
from, and the zero-point error is separated from it rather than confused with it. This is the
whole argument for whole-profile refinement over peak-position fitting: every point of the
profile contributes, so the answer is constrained by the shape of the pattern and not only by
where a handful of maxima happen to land.

The standard plot of a refinement is four curves at once — observed, calculated, background, and
the difference on its own axis underneath. The difference curve is the most informative single
output; structure in it is unmodelled physics.

In [ ]:
fig, (top, bottom) = plt.subplots(2, 1, figsize=(9.5, 6.2), sharex=True,
                                  gridspec_kw={"height_ratios": [3, 1]})
top.plot(result.two_theta_deg, result.observed_intensity, lw=0.0, marker="o", ms=1.2,
         color="#c0392b", label="observed")
top.plot(result.two_theta_deg, result.calculated_intensity, lw=1.0, color="#1f4e79",
         label="calculated")
top.plot(result.two_theta_deg, result.background_intensity, lw=1.0, color="#2e7d32",
         label="refined background")
for reflection in result.reflections:
    top.axvline(reflection.two_theta_deg, ymax=0.04, color="#111111", lw=0.8)
top.set_ylabel("counts")
top.set_yscale("log")
top.set_ylim(80.0, 5.0e4)
top.legend(loc="upper right", fontsize=8)
top.set_title(
    f"Rietveld refinement of {result.phase.name}: "
    f"$R_{{wp}}$ = {100 * result.weighted_profile_r_factor:.2f}%, "
    f"GoF = {result.goodness_of_fit:.3f}"
)

bottom.axhline(0.0, color="#111111", lw=0.8)
bottom.plot(result.two_theta_deg, result.residual_intensity, lw=0.6, color="#6a1b9a")
bottom.set_xlabel(r"$2\theta$ (deg)")
bottom.set_ylabel("obs - calc")
bottom.set_xlim(30.0, 130.0)
fig.tight_layout()

## 7. Reading the agreement indices honestly

Four numbers, and the relationship between them matters more than any one of them.

- $R_p = \sum|y_o - y_c| / \sum y_o$ — the plain profile residual.
- $R_{wp} = \sqrt{\sum w(y_o-y_c)^2 / \sum w y_o^2}$ — the quantity actually minimized.
- $R_{exp} = \sqrt{(N-P)/\sum w y_o^2}$ — the value $R_{wp}$ would take if the *only* remaining
  misfit were counting noise. It is a property of the data, not of the model.
- $\chi^2 = (R_{wp}/R_{exp})^2$, the goodness of fit.

$R_{wp}$ on its own is nearly uninterpretable, because it depends on how much background the scan
contains: the same model on the same specimen gives a much lower $R_{wp}$ if you collect a scan
with a large flat background, since the background is easy to fit and inflates the denominator.
The ratio to $R_{exp}$ is what removes that. Toby (2006) is the standard statement of this.

In [ ]:
print(f"R_p     {100 * result.profile_r_factor:7.3f} %")
print(f"R_wp    {100 * result.weighted_profile_r_factor:7.3f} %")
print(f"R_exp   {100 * result.expected_r_factor:7.3f} %   <- the counting-noise floor")
print(f"R_Bragg {100 * result.bragg_r_factor:7.3f} %")
print(f"GoF     {result.goodness_of_fit:7.3f}     (chi^2 = {result.reduced_chi_squared:.3f})")
print(f"Durbin-Watson {result.durbin_watson:.3f}")
print()
print(f"{result.point_count} observations, {result.refined_parameter_count} refined parameters, "
      f"weights: {result.weight_model}")

A goodness of fit near 1 says the calculated pattern explains the data down to the counting noise
and no further, which is the best any model can do. **Below 1 is not better** — it means the model
is following noise, either because the weights are too pessimistic or because too many parameters
are being refined.

The Durbin–Watson statistic asks a different question: are neighbouring residuals independent?
Values near 2 say yes. Values well below 1 say the misfit is *systematic* — a peak-shape error, an
unmodelled phase — because a small error repeated consistently across a peak is much worse
evidence against a model than the same total error scattered randomly. It is the statistic that
still discriminates when $R_{wp}$ has been flattered by a large background, which is the case a
single agreement index handles worst.

Here is what that looks like. Refine the same data with the peak width held fixed at the wrong
value. Every index degrades — but they do not degrade equally, and the two that move furthest are
the two that are sensitive to the *shape* of the misfit rather than its size: $R_{Bragg}$, which
compares partitioned integrated intensities, and Durbin–Watson, which collapses towards zero.

In [ ]:
crippled = refine_rietveld(
    measured,
    NICKEL,
    radiation=RADIATION,
    instrument=InstrumentBroadening.ideal(0.10),
    refine=("scale", "zero_shift_deg", "lattice_scale"),
    background_degree=10,
    intensity_model="xray_tabulated",
)
print(f"{'':<26}{'width refined':>16}{'width fixed wrong':>20}")
print("-" * 62)
for label, good, bad in (
    ("R_wp (%)", 100 * result.weighted_profile_r_factor,
     100 * crippled.weighted_profile_r_factor),
    ("goodness of fit", result.goodness_of_fit, crippled.goodness_of_fit),
    ("Durbin-Watson", result.durbin_watson, crippled.durbin_watson),
    ("R_Bragg (%)", 100 * result.bragg_r_factor, 100 * crippled.bragg_r_factor),
):
    print(f"{label:<26}{good:>16.3f}{bad:>20.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(9.0, 3.4))
window = (result.two_theta_deg > 46.0) & (result.two_theta_deg < 49.0)
ax.axhline(0.0, color="#111111", lw=0.8)
ax.plot(result.two_theta_deg[window], result.residual_intensity[window], lw=1.2,
        color="#1f4e79", label="width refined")
ax.plot(crippled.two_theta_deg[window], crippled.residual_intensity[window], lw=1.2,
        color="#c0392b", label="width fixed at the wrong value")
ax.set_xlabel(r"$2\theta$ (deg)")
ax.set_ylabel("obs - calc")
ax.set_title("The 022 reflection: a systematic residual has a shape, and it is the peak's")
ax.legend(loc="upper right", fontsize=8)
fig.tight_layout()

The wrong-width residual is the classic derivative-shaped signature: a trough at the peak centre
flanked by two crests, because a calculated peak that is too narrow is too tall in the middle and
too low in the wings. It repeats at *every* reflection, which is what drives the Durbin–Watson
statistic down and what no single agreement index would have told you.

## 8. Texture

A pressed or rolled specimen is not a random powder, and the intensity ratios say so before
anything else does. March–Dollase describes the commonest case — fibre texture about one specimen
axis — with a single parameter $r$: below 1 is plate-like and enhances the preferred reflections,
above 1 is needle-like and suppresses them, and exactly 1 is a random powder.

Refining it needs the axis stated, because a texture strength about an unstated axis has no
meaning, and the API refuses rather than guessing.

In [ ]:
try:
    refine_rietveld(measured, NICKEL, radiation=RADIATION,
                    refine=("scale", "march_coefficient"))
except ValueError as error:
    print(f"refused: {error}")

In [ ]:
TRUE_MARCH = 0.75

texture_model = MarchDollaseModel(
    preferred_orientation=MillerPlane(
        indices=np.array([1, 1, 1], dtype=np.int64), phase=NICKEL
    ),
    march_coefficient=TRUE_MARCH,
)
textured_pattern = generate_xrd_pattern(
    NICKEL,
    radiation=RADIATION,
    two_theta_range_deg=(30.0, 130.0),
    resolution_deg=0.02,
    broadening_fwhm_deg=TRUE_FWHM_DEG,
    intensity_model="xray_tabulated",
    preferred_orientation=texture_model,
)
textured = MeasuredPowderPattern(
    name="nickel with (111) fibre texture",
    two_theta_deg=textured_pattern.two_theta_grid_deg,
    intensity=np.random.default_rng(11).poisson(
        20000.0 * textured_pattern.intensity_grid + 150.0
    ).astype(float),
    radiation=RADIATION,
    synthetic=True,
)

textured_result = refine_rietveld(
    textured,
    NICKEL,
    radiation=RADIATION,
    instrument=InstrumentBroadening.ideal(0.10),
    refine=("scale", "caglioti_w", "march_coefficient"),
    preferred_orientation_plane=(1, 1, 1),
    background_degree=3,
    intensity_model="xray_tabulated",
)
march = textured_result.parameter("march_coefficient")
print(f"refined r = {march.format()}   (true {TRUE_MARCH})")
assert abs(march.value - TRUE_MARCH) < 0.03

Worth knowing about that number: $r = 1$ is a *bad place to start a refinement from*, because
plate-like and needle-like textures lie on opposite sides of it and a local optimizer commits to
whichever side its first derivative happens to point. On this very pattern, started at exactly 1,
a plain least-squares settles at $r \approx 1.26$ while the true 0.75 was available at half the
$R_{wp}$. PyTex therefore runs a short coarse scan over $r$ before refining — but only when the
caller has left the default, since a stated starting value is information the scan would discard.

Local minima in texture parameters are not an artefact of this implementation. They are a
standing hazard of Rietveld refinement, and the general defence is the same: start from a value
you have a reason for, and look at the difference curve rather than the R factor.

## 9. What is deliberately not refined, and why

PyTex will not refine atomic coordinates, site occupancies, anisotropic displacement parameters,
or the independent cell edges of a low-symmetry cell. That is a design decision, not a gap.

Those are the parameters that turn a refinement into a *structure determination*, and they need
apparatus this module does not have: constraints tying symmetry-related sites, restraints on bond
lengths and angles, rigid-body or symmetry-mode parameterizations, and a way to detect that two
parameters have become indistinguishable. Without that apparatus they will still refine — they
will produce a lower $R_{wp}$ and a structure nobody should publish. A tool that makes the wrong
answer easy to reach and hard to notice is worse than one that declines.

What is offered instead is the part that is well posed against a *known* structure: the cell, the
zero point, the peak shape, the texture strength, the background, and an honest account of how
well the model does. `describe()` states that scope in the output itself, so a reader of a result
cannot mistake it for something larger.

One protection worth knowing about. Standard uncertainties come from the singular value
decomposition of the Jacobian, and a rank-deficient pair — two parameters the data genuinely
cannot tell apart — gets an *infinite* uncertainty rather than a small one. A correlated pair is
therefore visible in the output instead of being quietly reported as well determined.

In [ ]:
print("What a refinement result carries, and what it is for:\n")
for label, value in (
    ("phase, as refined", f"{result.phase.name}, a = {result.phase.lattice.a:.6f} A"),
    ("profile arrays", f"{result.point_count} points, observed/calculated/background/residual"),
    ("reflection list", f"{len(result.reflections)} families, indexed and attributable"),
    ("parameters", f"{len(result.parameters)} total, {result.refined_parameter_count} refined"),
    ("agreement", f"R_wp {100 * result.weighted_profile_r_factor:.2f}%, "
                  f"GoF {result.goodness_of_fit:.3f}"),
    ("as_pattern()", "the calculated Bragg profile, background removed, as a PowderPattern"),
    ("describe()", "the whole of the above as citation-backed prose"),
):
    print(f"  {label:<20} {value}")

## Summary

| Question | Tool | The thing to remember |
| --- | --- | --- |
| What is the background? | `estimate_background` | SNIP to look at it, Chebyshev to refine it — and do not subtract before refining |
| How wide is the instrument? | `calibrate_instrument_broadening` | Linear in $U$, $V$, $W$, so the fit is exact; calibrate from a standard, never guess |
| How large are the crystallites? | `deconvolve_instrument_width`, `williamson_hall` | One peak cannot separate size from strain; several peaks can, because their angular dependences differ |
| Is this the phase? | `compare_powder_patterns` | Fits a scale, moves nothing — identification, not measurement |
| What does the pattern say the cell is? | `refine_rietveld` | Refine the zero point too, or the cell will absorb it |
| Is the fit any good? | `RietveldResult.describe()` | $R_{wp}$ alone means little; use $R_{wp}/R_{exp}$, and read the difference curve |

### See also

- Tutorial 11, `11_powder_xrd_workflows` — the simulation this one starts from.
- `docs/architecture/application_platform.md` — the same operations in the workbench's XRD panel.
- Rietveld, *J. Appl. Crystallogr.* **2** (1969) 65, [doi:10.1107/S0021889869006558](https://doi.org/10.1107/S0021889869006558).
- Toby, *Powder Diffr.* **21** (2006) 67, [doi:10.1154/1.2179804](https://doi.org/10.1154/1.2179804) — how good is good enough.
- Ryan *et al.*, *Nucl. Instrum. Methods B* **34** (1988) 396, [doi:10.1016/0168-583X(88)90063-8](https://doi.org/10.1016/0168-583X(88)90063-8) — SNIP.
- Caglioti, Paoletti & Ricci, *Nucl. Instrum.* **3** (1958) 223, [doi:10.1016/0369-643X(58)90029-X](https://doi.org/10.1016/0369-643X(58)90029-X).
- Thompson, Cox & Hastings, *J. Appl. Crystallogr.* **20** (1987) 79, [doi:10.1107/S0021889887087090](https://doi.org/10.1107/S0021889887087090).
- Williamson & Hall, *Acta Metall.* **1** (1953) 22, [doi:10.1016/0001-6160(53)90006-6](https://doi.org/10.1016/0001-6160(53)90006-6).